In [ ]:
import os
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from itertools import combinations
import re
from collections import defaultdict
from sklearn.model_selection import train_test_split
from tqdm import tqdm
tqdm.pandas()

import copy
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

df_final = pd.read_csv('../data/df_final_prepared.csv')

In [ ]:
print("СЛОВАРИ ДЛЯ ИЕРАРХИИ")

# ТОВАРЫ
items = sorted(df_final['name_clean'].unique())
item_to_idx = {x: i for i, x in enumerate(items)}

# УРОВЕНЬ 2
lvl2 = sorted(df_final['art_grp_lvl_2_name'].unique())
lvl2_to_idx = {x: i for i, x in enumerate(lvl2)}

# УРОВЕНЬ 1
lvl1 = sorted(df_final['art_grp_lvl_1_name'].unique())
lvl1_to_idx = {x: i for i, x in enumerate(lvl1)}

# УРОВЕНЬ 0
lvl0 = sorted(df_final['art_grp_lvl_0_name'].unique())
lvl0_to_idx = {x: i for i, x in enumerate(lvl0)}

# СТАТИСТИКА
print("\nРазмерности:")
print(f"Items: {len(items)}")
print(f"Level 2: {len(lvl2)}")
print(f"Level 1: {len(lvl1)}")
print(f"Level 0: {len(lvl0)}")

# САНИТИ ЧЕК
print("\nПример:")
sample = df_final.iloc[0]

print("\nТовар:")
print(sample['name_clean'])

print("\nИерархия:")
print("lvl2:", sample['art_grp_lvl_2_name'])
print("lvl1:", sample['art_grp_lvl_1_name'])
print("lvl0:", sample['art_grp_lvl_0_name'])

СЛОВАРИ ДЛЯ ИЕРАРХИИ

Размерности:
Items: 27212
Level 2: 631
Level 1: 185
Level 0: 36

Пример:

Товар:
коньяк армянский аревик 5лет 40  0 5л под уп армения  6

Иерархия:
lvl2: Коньяк Армения
lvl1: Коньяк
lvl0: Крепкий алкоголь


In [ ]:
print("MULTI-VIEW BASKETS")

# группируем
baskets = df_final.groupby('cheque_id').agg({
    'name_clean': list,
    'art_grp_lvl_2_name': list,
    'art_grp_lvl_1_name': list,
    'art_grp_lvl_0_name': list
}).reset_index()

print(f"\nВсего корзин: {len(baskets)}")

# функции создания multi-hot

def build_vector(elements, mapping):
    vec = np.zeros(len(mapping), dtype=np.float32)
    for e in elements:
        if e in mapping:
            vec[mapping[e]] = 1.0
    return vec

# создаём все представления
item_vectors = []
lvl2_vectors = []
lvl1_vectors = []
lvl0_vectors = []

for i, row in baskets.iterrows():
    
    if i % 10000 == 0:
        print(f"Обработано: {i}")
    
    item_vectors.append(build_vector(row['name_clean'], item_to_idx))
    lvl2_vectors.append(build_vector(row['art_grp_lvl_2_name'], lvl2_to_idx))
    lvl1_vectors.append(build_vector(row['art_grp_lvl_1_name'], lvl1_to_idx))
    lvl0_vectors.append(build_vector(row['art_grp_lvl_0_name'], lvl0_to_idx))


# в numpy
item_vectors = np.array(item_vectors, dtype=np.float32)
lvl2_vectors = np.array(lvl2_vectors, dtype=np.float32)
lvl1_vectors = np.array(lvl1_vectors, dtype=np.float32)
lvl0_vectors = np.array(lvl0_vectors, dtype=np.float32)

# проверка
print("\nShapes:")
print("item:", item_vectors.shape)
print("lvl2:", lvl2_vectors.shape)
print("lvl1:", lvl1_vectors.shape)
print("lvl0:", lvl0_vectors.shape)

MULTI-VIEW BASKETS

Всего корзин: 49337
Обработано: 0
Обработано: 10000
Обработано: 20000
Обработано: 30000
Обработано: 40000

Shapes:
item: (49337, 27212)
lvl2: (49337, 631)
lvl1: (49337, 185)
lvl0: (49337, 36)


In [ ]:
print("TRAIN / VAL / TEST SPLIT")

cheque_ids = baskets['cheque_id'].values

# 70 / 15 / 15
train_idx, temp_idx = train_test_split(
    np.arange(len(baskets)),
    test_size=0.30,
    random_state=42,
    shuffle=True
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

# ITEM
item_train = item_vectors[train_idx]
item_val   = item_vectors[val_idx]
item_test  = item_vectors[test_idx]

# LVL2
lvl2_train = lvl2_vectors[train_idx]
lvl2_val   = lvl2_vectors[val_idx]
lvl2_test  = lvl2_vectors[test_idx]

# LVL1
lvl1_train = lvl1_vectors[train_idx]
lvl1_val   = lvl1_vectors[val_idx]
lvl1_test  = lvl1_vectors[test_idx]

# LVL0
lvl0_train = lvl0_vectors[train_idx]
lvl0_val   = lvl0_vectors[val_idx]
lvl0_test  = lvl0_vectors[test_idx]

# IDS
train_ids = cheque_ids[train_idx]
val_ids   = cheque_ids[val_idx]
test_ids  = cheque_ids[test_idx]

# СТАТИСТИКА
print("\nРазмеры split:")
print(f"Train: {len(train_idx):,}")
print(f"Val:   {len(val_idx):,}")
print(f"Test:  {len(test_idx):,}")

print("\nShapes:")
print("item_train:", item_train.shape)
print("item_val:  ", item_val.shape)
print("item_test: ", item_test.shape)

print("lvl2_train:", lvl2_train.shape)
print("lvl2_val:  ", lvl2_val.shape)
print("lvl2_test: ", lvl2_test.shape)

print("lvl1_train:", lvl1_train.shape)
print("lvl1_val:  ", lvl1_val.shape)
print("lvl1_test: ", lvl1_test.shape)

print("lvl0_train:", lvl0_train.shape)
print("lvl0_val:  ", lvl0_val.shape)
print("lvl0_test: ", lvl0_test.shape)

# SANITY CHECK
print("\nПример cheque_id:")
print("train:", train_ids[:3])
print("val:  ", val_ids[:3])
print("test: ", test_ids[:3])

TRAIN / VAL / TEST SPLIT

Размеры split:
Train: 34,535
Val:   7,401
Test:  7,401

Shapes:
item_train: (34535, 27212)
item_val:   (7401, 27212)
item_test:  (7401, 27212)
lvl2_train: (34535, 631)
lvl2_val:   (7401, 631)
lvl2_test:  (7401, 631)
lvl1_train: (34535, 185)
lvl1_val:   (7401, 185)
lvl1_test:  (7401, 185)
lvl0_train: (34535, 36)
lvl0_val:   (7401, 36)
lvl0_test:  (7401, 36)

Пример cheque_id:
train: [5.77406656e+09 5.95026744e+09 6.01398598e+09]
val:   [5.99664596e+09 5.96636946e+09 6.04712559e+09]
test:  [5.99633934e+09 5.98732338e+09 5.85122529e+09]


In [ ]:
# DATASET ДЛЯ MULTI-VIEW VAE

class MultiViewDataset(Dataset):
    def __init__(self, item, lvl2, lvl1, lvl0):
        self.item = item
        self.lvl2 = lvl2
        self.lvl1 = lvl1
        self.lvl0 = lvl0
        
    def __len__(self):
        return len(self.item)
    
    def __getitem__(self, idx):
        return {
            'item': torch.FloatTensor(self.item[idx]),
            'lvl2': torch.FloatTensor(self.lvl2[idx]),
            'lvl1': torch.FloatTensor(self.lvl1[idx]),
            'lvl0': torch.FloatTensor(self.lvl0[idx]),
        }


# создаём datasets
train_dataset = MultiViewDataset(
    item_train, lvl2_train, lvl1_train, lvl0_train
)

val_dataset = MultiViewDataset(
    item_val, lvl2_val, lvl1_val, lvl0_val
)

print("Dataset создан")
print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")

# проверка
sample = train_dataset[0]

print("\nSample shapes:")
for k, v in sample.items():

    print(k, v.shape)

Dataset создан
Train size: 34535
Val size: 7401

Sample shapes:
item torch.Size([27212])
lvl2 torch.Size([631])
lvl1 torch.Size([185])
lvl0 torch.Size([36])


In [ ]:
print("ПОДГОТОВКА КОНТЕКСТА ДЛЯ EVALUATION VAE")

# маппинг товаров - категории
item_meta = (
    df_final[['name_clean', 'art_grp_lvl_2_name', 'art_grp_lvl_1_name', 'art_grp_lvl_0_name']]
    .drop_duplicates(subset=['name_clean'])
    .reset_index(drop=True)
)

item_to_lvl2 = dict(zip(item_meta['name_clean'], item_meta['art_grp_lvl_2_name']))
item_to_lvl1 = dict(zip(item_meta['name_clean'], item_meta['art_grp_lvl_1_name']))
item_to_lvl0 = dict(zip(item_meta['name_clean'], item_meta['art_grp_lvl_0_name']))

print(f"Маппинги созданы для {len(item_to_lvl2):,} товаров")

# ФУНКЦИЯ: ИЗ СПИСКА ТОВАРОВ СОБРАТЬ 4 ВЕКТОРА
def build_multiview_from_items(item_names):
    x_item = np.zeros(len(item_to_idx), dtype=np.float32)
    x_lvl2 = np.zeros(len(lvl2_to_idx), dtype=np.float32)
    x_lvl1 = np.zeros(len(lvl1_to_idx), dtype=np.float32)
    x_lvl0 = np.zeros(len(lvl0_to_idx), dtype=np.float32)
    
    for item in item_names:
        if item in item_to_idx:
            x_item[item_to_idx[item]] = 1.0
        
        if item in item_to_lvl2 and item_to_lvl2[item] in lvl2_to_idx:
            x_lvl2[lvl2_to_idx[item_to_lvl2[item]]] = 1.0
        
        if item in item_to_lvl1 and item_to_lvl1[item] in lvl1_to_idx:
            x_lvl1[lvl1_to_idx[item_to_lvl1[item]]] = 1.0
        
        if item in item_to_lvl0 and item_to_lvl0[item] in lvl0_to_idx:
            x_lvl0[lvl0_to_idx[item_to_lvl0[item]]] = 1.0
    
    return x_item, x_lvl2, x_lvl1, x_lvl0

# СОБЕРЁМ TEST-КОРЗИНЫ В ВИДЕ СПИСКОВ ТОВАРОВ
test_baskets_df = baskets.iloc[test_idx].copy()
val_baskets_df = baskets.iloc[val_idx].copy()

print(f"Test корзин: {len(test_baskets_df):,}")

sample_items = test_baskets_df.iloc[0]['name_clean'][:5]
print("\nПример товаров из test корзины:")
print(sample_items)

sample_x = build_multiview_from_items(sample_items)

print("\nShapes sample multiview:")
print("item:", sample_x[0].shape, "active:", int(sample_x[0].sum()))
print("lvl2:", sample_x[1].shape, "active:", int(sample_x[1].sum()))
print("lvl1:", sample_x[2].shape, "active:", int(sample_x[2].sum()))
print("lvl0:", sample_x[3].shape, "active:", int(sample_x[3].sum()))

ПОДГОТОВКА КОНТЕКСТА ДЛЯ EVALUATION VAE
Маппинги созданы для 27,212 товаров
Test корзин: 7,401

Пример товаров из test корзины:
['gillette 2 одноразовые станки 10шт  проктер  12 24', 'пиво светлое жигулевское паст 4  1 25л пл б  балтика  9', 'томаты черри 250г', 'monarch original кофе нат раств 190г ст бан 6', 'mentos жевательная резинка виноград 15 5г ван мелле  24']

Shapes sample multiview:
item: (27212,) active: 5
lvl2: (631,) active: 5
lvl1: (185,) active: 5
lvl0: (36,) active: 5


In [8]:
# RECALL
def recall_vae(model, baskets, k=10, n=3000, seed=42):
    rng = np.random.default_rng(seed)
    
    indices = rng.choice(len(baskets), size=n, replace=False)
    
    hits = 0
    
    for i in tqdm(indices):
        items = list(baskets.iloc[i]['name_clean'])
        items = list(dict.fromkeys(items))
        
        if len(items) < 2:
            continue
        
        target = rng.choice(items)
        context = [x for x in items if x != target]
        
        x = build_multiview_from_items(context)
        
        x = {
            'item': torch.FloatTensor(x[0]).unsqueeze(0).to(device),
            'lvl2': torch.FloatTensor(x[1]).unsqueeze(0).to(device),
            'lvl1': torch.FloatTensor(x[2]).unsqueeze(0).to(device),
            'lvl0': torch.FloatTensor(x[3]).unsqueeze(0).to(device)
        }
        
        with torch.no_grad():
            recon, _, _ = model(x)
            scores = recon['item'].cpu().numpy().flatten()
        
        context_idx = [item_to_idx[x] for x in context if x in item_to_idx]
        scores[context_idx] = -np.inf
        
        top = np.argpartition(-scores, k)[:k]
        
        if target in item_to_idx and item_to_idx[target] in top:
            hits += 1
    
    return hits / n

In [ ]:
class MultiViewDataset(Dataset):
    def __init__(self, item, lvl2, lvl1, lvl0):
        self.item = item
        self.lvl2 = lvl2
        self.lvl1 = lvl1
        self.lvl0 = lvl0
        
    def __len__(self):
        return len(self.item)
    
    def __getitem__(self, idx):
        return {
            'item': torch.FloatTensor(self.item[idx]),
            'lvl2': torch.FloatTensor(self.lvl2[idx]),
            'lvl1': torch.FloatTensor(self.lvl1[idx]),
            'lvl0': torch.FloatTensor(self.lvl0[idx]),
        }

train_dataset = MultiViewDataset(item_train, lvl2_train, lvl1_train, lvl0_train)
val_dataset   = MultiViewDataset(item_val, lvl2_val, lvl1_val, lvl0_val)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=128, shuffle=False)

# MODEL
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n_items = item_train.shape[1]
n_lvl2 = lvl2_train.shape[1]
n_lvl1 = lvl1_train.shape[1]
n_lvl0 = lvl0_train.shape[1]

class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.item_enc = nn.Sequential(
            nn.Linear(n_items, 512),
            nn.ReLU(),
            nn.Linear(512, 256)
        )
        self.lvl2_enc = nn.Sequential(
            nn.Linear(n_lvl2, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )
        self.lvl1_enc = nn.Sequential(
            nn.Linear(n_lvl1, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )
        self.lvl0_enc = nn.Sequential(
            nn.Linear(n_lvl0, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )
        
        self.fc_mu = nn.Linear(256 + 64 + 32 + 16, 64)
        self.fc_logvar = nn.Linear(256 + 64 + 32 + 16, 64)
        
        self.item_dec = nn.Linear(64, n_items)
        self.lvl2_dec = nn.Linear(64, n_lvl2)
        self.lvl1_dec = nn.Linear(64, n_lvl1)
        self.lvl0_dec = nn.Linear(64, n_lvl0)
    
    def encode(self, x):
        h = torch.cat([
            self.item_enc(x['item']),
            self.lvl2_enc(x['lvl2']),
            self.lvl1_enc(x['lvl1']),
            self.lvl0_enc(x['lvl0'])
        ], dim=1)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparam(self, mu, logvar, sample=True):
        if not sample:
            return mu
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return {
            'item': self.item_dec(z),
            'lvl2': self.lvl2_dec(z),
            'lvl1': self.lvl1_dec(z),
            'lvl0': self.lvl0_dec(z),
        }
    
    def forward(self, x, sample=True):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar, sample=sample)
        recon = self.decode(z)
        return recon, mu, logvar

model = VAE().to(device)

# LOSS
def loss_fn(recon, x, mu, logvar, beta):
    bce_item = F.binary_cross_entropy_with_logits(
        recon['item'], x['item'], reduction='sum'
    )
    bce_lvl2 = F.binary_cross_entropy_with_logits(
        recon['lvl2'], x['lvl2'], reduction='sum'
    )
    bce_lvl1 = F.binary_cross_entropy_with_logits(
        recon['lvl1'], x['lvl1'], reduction='sum'
    )
    bce_lvl0 = F.binary_cross_entropy_with_logits(
        recon['lvl0'], x['lvl0'], reduction='sum'
    )

    bce = (
        0.7 * bce_item +
        1.0 * bce_lvl2 +
        0.8 * bce_lvl1 +
        0.6 * bce_lvl0
    )
    
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    total = bce + beta * kl
    
    return total, bce, kl

# MASKING
def mask_numpy_row(x, p=0.3):
    x = x.copy()
    idx = np.where(x == 1)[0]
    if len(idx) > 1:
        n_drop = max(1, int(len(idx) * p))
        drop = np.random.choice(idx, n_drop, replace=False)
        x[drop] = 0
    return x

def apply_train_mask(item_tensor, p=0.3):
    masked_np = np.array(
        [mask_numpy_row(x, p=p) for x in item_tensor.cpu().numpy()],
        dtype=np.float32
    )
    return torch.from_numpy(masked_np).to(item_tensor.device)

# TRAIN SETUP
opt = optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 50
beta = 0.4
train_mask_p = 0.3

patience = 5
min_delta = 1e-4
counter = 0

best_val_recall = -1.0
best_epoch = -1
best_state = None

history = {
    'train_loss': [],
    'train_bce': [],
    'train_kl': [],
    'val_loss': [],
    'val_bce': [],
    'val_kl': [],
    'val_recall@10': [],
}

# TRAIN LOOP
for epoch in range(1, n_epochs + 1):
    model.train()

    train_total_loss = 0.0
    train_total_bce = 0.0
    train_total_kl = 0.0
    train_n = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{n_epochs}", leave=False)

    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        bs = batch['item'].size(0)

        masked = {k: v.clone() for k, v in batch.items()}
        masked['item'] = apply_train_mask(batch['item'], p=train_mask_p)

        recon, mu, logvar = model(masked, sample=True)
        loss, bce, kl = loss_fn(recon, batch, mu, logvar, beta)

        opt.zero_grad()
        loss.backward()
        opt.step()

        train_total_loss += loss.item()
        train_total_bce += bce.item()
        train_total_kl += kl.item()
        train_n += bs

        pbar.set_postfix({
            "loss/batch": f"{loss.item()/bs:.2f}",
            "bce/batch": f"{bce.item()/bs:.2f}",
            "kl/batch": f"{kl.item()/bs:.2f}",
            "beta": f"{beta:.2f}",
        })

    train_loss = train_total_loss / train_n
    train_bce  = train_total_bce / train_n
    train_kl   = train_total_kl / train_n

    # VALIDATION LOSS
    model.eval()

    val_total_loss = 0.0
    val_total_bce = 0.0
    val_total_kl = 0.0
    val_n = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            bs = batch['item'].size(0)

            recon, mu, logvar = model(batch, sample=False)
            loss, bce, kl = loss_fn(recon, batch, mu, logvar, beta)

            val_total_loss += loss.item()
            val_total_bce += bce.item()
            val_total_kl += kl.item()
            val_n += bs

    val_loss = val_total_loss / val_n
    val_bce  = val_total_bce / val_n
    val_kl   = val_total_kl / val_n

    # VALIDATION RECALL 
    val_recall = recall_vae(
        model=model,
        baskets=val_baskets_df,
        k=10,
        n=min(10000, len(val_baskets_df)),
        seed=42
    )

    history['train_loss'].append(train_loss)
    history['train_bce'].append(train_bce)
    history['train_kl'].append(train_kl)
    history['val_loss'].append(val_loss)
    history['val_bce'].append(val_bce)
    history['val_kl'].append(val_kl)
    history['val_recall@10'].append(val_recall)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss: {train_loss:.4f} | "
        f"train_bce: {train_bce:.4f} | "
        f"train_kl: {train_kl:.4f} | "
        f"val_loss: {val_loss:.4f} | "
        f"val_bce: {val_bce:.4f} | "
        f"val_kl: {val_kl:.4f} | "
        f"val_recall@10: {val_recall:.4f}"
    )

    # EARLY STOPPING BY VALIDATION RECALL
    if val_recall > best_val_recall + min_delta:
        best_val_recall = val_recall
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        counter = 0
        print(f"- new best model saved (epoch {epoch}, val_recall@10={val_recall:.4f})")
    else:
        counter += 1
        print(f"- no recall improvement, early stopping counter: {counter}/{patience}")

    if counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

# RESTORE BEST MODEL
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Best model restored from epoch {best_epoch} with val_recall@10={best_val_recall:.4f}")
else:
    print("Warning: best_state is None, model was not restored.")


100%|██████████| 7401/7401 [00:58<00:00, 127.03it/s]                                                                            


Epoch 01 | train_loss: 1277.5482 | train_bce: 1104.5736 | train_kl: 432.4366 | val_loss: 264.5870 | val_bce: 183.9442 | val_kl: 201.6071 | val_recall@10: 0.0739
- new best model saved (epoch 1, val_recall@10=0.0739)


100%|██████████| 7401/7401 [00:56<00:00, 131.41it/s]                                                                          


Epoch 02 | train_loss: 244.4927 | train_bce: 176.6857 | train_kl: 169.5175 | val_loss: 221.7702 | val_bce: 168.5971 | val_kl: 132.9327 | val_recall@10: 0.0767
- new best model saved (epoch 2, val_recall@10=0.0767)


100%|██████████| 7401/7401 [00:56<00:00, 130.43it/s]                                                                          


...

100%|██████████| 7401/7401 [00:54<00:00, 135.58it/s]                                                                          


Epoch 16 | train_loss: 153.6676 | train_bce: 134.8296 | train_kl: 47.0951 | val_loss: 146.8672 | val_bce: 128.5905 | val_kl: 45.6919 | val_recall@10: 0.0797
- no recall improvement, early stopping counter: 4/5


100%|██████████| 7401/7401 [00:57<00:00, 128.65it/s]                                                                          


Epoch 17 | train_loss: 150.7652 | train_bce: 131.4075 | train_kl: 48.3943 | val_loss: 143.9796 | val_bce: 124.5882 | val_kl: 48.4783 | val_recall@10: 0.0788
- no recall improvement, early stopping counter: 5/5
Early stopping at epoch 17
Best model restored from epoch 12 with val_recall@10=0.0892


100%|██████████| 7401/7401 [00:57<00:00, 129.66it/s]


In [ ]:
recalls = []

for _ in range(5):
    rec = recall_vae(model, test_baskets_df, n=len(test_baskets_df))
    recalls.append(rec)

print(f"FINAL VAE Recall@10: {np.mean(recalls)}")

FINAL VAE Recall@10: 0.0862045669504121


In [ ]:
print("СОХРАНЕНИЕ VAE МОДЕЛИ")

save_dir = "../data/vae_final"
os.makedirs(save_dir, exist_ok=True)

torch.save(model.state_dict(), f"{save_dir}/vae_model.pt")

print("модель сохранена: vae_model.pt")

config = {
    "latent_dim": 64,
    "beta": beta,
    "mask_p": train_mask_p,
    "best_epoch": best_epoch,
    "val_recall": best_val_recall
}

with open(f"{save_dir}/config.pkl", "wb") as f:
    pickle.dump(config, f)

print("config сохранён")

with open(f"{save_dir}/item_to_idx.pkl", "wb") as f:
    pickle.dump(item_to_idx, f)

with open(f"{save_dir}/lvl2_to_idx.pkl", "wb") as f:
    pickle.dump(lvl2_to_idx, f)

with open(f"{save_dir}/lvl1_to_idx.pkl", "wb") as f:
    pickle.dump(lvl1_to_idx, f)

with open(f"{save_dir}/lvl0_to_idx.pkl", "wb") as f:
    pickle.dump(lvl0_to_idx, f)

print("mapping-и сохранены")

idx_to_item = {v: k for k, v in item_to_idx.items()}

with open(f"{save_dir}/idx_to_item.pkl", "wb") as f:
    pickle.dump(idx_to_item, f)

print("обратные mapping-и сохранены")

print(f"FINAL VAE Recall@10: {recalls:.6f}")

СОХРАНЕНИЕ VAE МОДЕЛИ
модель сохранена: vae_model.pt
config сохранён
mapping-и сохранены
обратные mapping-и сохранены
FINAL VAE Recall@10: 0.086205
